# Immunogenicity tool test: Testing setttings for tools: IApred
Author: Rebecka Antonsson\
Version: 1 (04-03-2026)

In [1]:
import pandas as pd
from scipy.stats import spearmanr

In [2]:
IApred = pd.read_csv('tool_outputs/IApred_results.csv')
IApred.head()

,Header,Sequence_Length,IAScore,Antigenicity_Category
0,BEZLOTOXUMAB,227,0.57,High
1,VISILIZUMAB,226,0.88,High
2,OMALIZUMAB,232,0.47,High
3,EVOLOCUMAB,224,1.32,High
4,SECUKINUMAB,235,0.23,Moderate


In [3]:
# Create separate df without the two nanobodies
IApred_AB = IApred[~IApred['Header'].isin(['Caplacizumab', 'Vobarilizumab'])].copy()
IApred_AB.head()

,Header,Sequence_Length,IAScore,Antigenicity_Category
0,BEZLOTOXUMAB,227,0.57,High
1,VISILIZUMAB,226,0.88,High
2,OMALIZUMAB,232,0.47,High
3,EVOLOCUMAB,224,1.32,High
4,SECUKINUMAB,235,0.23,Moderate


In [4]:
# Load ADA ranking
ADA_rank = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'Caplacizumab':28,
'LANADELUMAB':29,
'BUROSUMAB':30,
'BENRALIZUMAB':31,
'ADALIMUMAB':32,
'IXEKIZUMAB':33,
'RITUXIMAB':34,
'INFLIXIMAB':35,
'GOLIMUMAB':36,
'Vobarilizumab':37,
'BOCOCIZUMAB':38,
'ALEMTUZUMAB':39
}

# create antoher dictionary where the two nanobdoies are removed 
ADA_rank_AB = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'LANADELUMAB':28,
'BUROSUMAB':29,
'BENRALIZUMAB':30,
'ADALIMUMAB':31,
'IXEKIZUMAB':32,
'RITUXIMAB':33,
'INFLIXIMAB':34,
'GOLIMUMAB':35,
'BOCOCIZUMAB':36,
'ALEMTUZUMAB':37
}

In [5]:
# Add ADA rank to the two dfs
IApred['ADA_rank'] = IApred['Header'].map(ADA_rank)
IApred_AB['ADA_rank'] = IApred_AB['Header'].map(ADA_rank_AB)

In [6]:
# Add a column with the rank from the tool, based on the IAScore
IApred['ranked_by_tool'] = IApred['IAScore'].rank(ascending=True, method='average')
IApred_AB['ranked_by_tool'] = IApred_AB['IAScore'].rank(ascending=True, method='average')


In [7]:
# Compute MARE for both dfs
IApred['MARE'] = IApred['ranked_by_tool'] - IApred['ADA_rank']
# for IApred_AB get the asbolute value of the difference between the ranked_by_tool and ADA_rank
IApred_AB['MARE'] = IApred_AB['ranked_by_tool'] - IApred_AB['ADA_rank']
IApred_AB['MARE'] = IApred_AB['MARE'].abs()

IApred_AB.head()

,Header,Sequence_Length,IAScore,Antigenicity_Category,ADA_rank,ranked_by_tool,MARE
0,BEZLOTOXUMAB,227,0.57,High,1,24.0,23.0
1,VISILIZUMAB,226,0.88,High,2,36.0,34.0
2,OMALIZUMAB,232,0.47,High,3,14.0,11.0
3,EVOLOCUMAB,224,1.32,High,4,37.0,33.0
4,SECUKINUMAB,235,0.23,Moderate,5,2.0,3.0


In [8]:
# # Create a ranked_score_table df with coulmns, datadframe 
# The sum of MARE, from the IApred_AB df
# The spearman rank correlation between ranked by tool and ADA rank, from the IApred_AB df
# The MARE for Caplacizumab and Vobarilizumab, fromt he IApred df
ranked_score_table = pd.DataFrame({
    'dataframe': ['IApred'],
    'sum_MARE': [IApred_AB['MARE'].sum()],
    'spearmanr': [spearmanr(IApred_AB['ranked_by_tool'], IApred_AB['ADA_rank']).correlation ],
    'spearmanr_pval': [spearmanr(IApred_AB['ranked_by_tool'], IApred_AB['ADA_rank']).pvalue ],
    'Caplacizumab_MARE': [IApred.loc[IApred['Header'] == 'Caplacizumab', 'MARE'].values[0] ],
    'Vobarilizumab_MARE': [IApred.loc[IApred['Header'] == 'Vobarilizumab', 'MARE'].values[0] ]
})

# save the ranked_score_table as a csv file
ranked_score_table.to_csv('ranked_score_table.csv', index=False)
ranked_score_table


,dataframe,sum_MARE,spearmanr,spearmanr_pval,Caplacizumab_MARE,Vobarilizumab_MARE
0,IApred,503.0,-0.209727,0.212821,-24.5,-13.0
